In [10]:
import xarray as xr
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import r2_score, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from statsmodels.stats.outliers_influence import variance_inflation_factor


In [11]:
min_lon, min_lat, max_lon, max_lat =[-61.0, -47.5, -60.0, -44.875]
min_time, max_time = pd.to_datetime("2013-01-01"), pd.to_datetime("2023-12-31")

fishing_ds = xr.open_dataset("../data/processed/targets/HKP_0.1.nc")
fishing = fishing_ds["PesoTotal"]/10000
fishing = fishing.fillna(0)

fishing_cpue_ds = xr.open_dataset("../data/processed/targets/cpue_HKP_0.1.nc")
fishing_cpue = fishing_cpue_ds["CPUE"]
fishing_cpue = fishing_cpue.fillna(0)

temp_ds = xr.open_dataset("../data/processed/dynamic/to_surface.nc")
temp = temp_ds["to"]
temp = (temp - temp.mean()) / temp.std()
temp = temp.fillna(0)

temp_bottom_ds = xr.open_dataset("../data/processed/dynamic/temp_bottom.nc")
temp_bottom_ds = temp_bottom_ds.rename({"to": "to_b"})
temp_bottom = temp_bottom_ds["to_b"]
temp_bottom = (temp_bottom - temp_bottom.mean()) / temp_bottom.std()
temp_bottom = temp_bottom.fillna(0)

chl_ds = xr.open_dataset("../data/processed/dynamic/chl.nc")
chl = chl_ds["CHL"]
chl = (chl - chl.mean()) / chl.std()
chl = chl.fillna(0)

mixed_ds = xr.open_dataset("../data/processed/dynamic/mixed_layer.nc")
mixed = mixed_ds["mlotst"]
mixed = (mixed - mixed.mean()) / mixed.std()
mixed = mixed.fillna(0)

depth_ds = xr.open_dataset("../data/processed/static/depth.nc")
depth = depth_ds["depth"] 
depth = (depth - depth.mean()) / depth.std()
depth = depth.fillna(0)
depth = depth.broadcast_like(temp)

zo_ds = xr.open_dataset("../data/processed/dynamic/zo_surface.nc")
zo = zo_ds["zo"]
zo = (zo - zo.mean()) / zo.std()
zo = zo.fillna(0)

so_ds = xr.open_dataset("../data/processed/dynamic/so_surface.nc")
so = so_ds["so"]
so = (so - so.mean()) / so.std()
so = so.fillna(0)

ugo_ds = xr.open_dataset("../data/processed/dynamic/ugo_surface.nc")
ugo = ugo_ds["ugo"]
ugo = (ugo - ugo.mean()) / ugo.std()
ugo = ugo.fillna(0)

vgo_ds = xr.open_dataset("../data/processed/dynamic/vgo_surface.nc")
vgo = vgo_ds["vgo"]
vgo = (vgo - vgo.mean()) / vgo.std()
vgo = vgo.fillna(0)

pp = xr.open_dataset("../data/processed/dynamic/pp.nc")
pp = pp["PP"]
pp = (pp - pp.mean()) / pp.std()
pp = pp.fillna(0)

cdm = xr.open_dataset("../data/processed/dynamic/cdm.nc")
cdm = cdm["CDM"]
cdm = (cdm - cdm.mean()) / cdm.std()
cdm = cdm.fillna(0)

spm = xr.open_dataset("../data/processed/dynamic/spm.nc")
spm = spm["SPM"]
spm = (spm - spm.mean()) / spm.std()    
spm = spm.fillna(0)

zsd = xr.open_dataset("../data/processed/dynamic/zsd.nc")
zsd = zsd["ZSD"]
zsd = (zsd - zsd.mean()) / zsd.std()
zsd = zsd.fillna(0)

maskB_ds = xr.open_dataset("../data/processed/static/fishing_area_mask_buffered_0.1.nc")
maskB = maskB_ds["mask"]
maskB = maskB.broadcast_like(fishing)

mask_ds = xr.open_dataset("../data/processed/static/fishing_area_mask_0.1.nc")
mask = mask_ds["mask"]
mask = mask.broadcast_like(fishing)

month = temp["time"].dt.month
month_sin = np.sin(2 * np.pi * month / 12)
month_cos = np.cos(2 * np.pi * month / 12)
month_sin = month_sin.broadcast_like(temp)
month_cos = month_cos.broadcast_like(temp)
month = month.broadcast_like(temp)

year = temp["time"].dt.year
year = (year - year.mean()) / year.std()
year = year.broadcast_like(temp)

lat = (temp["lat"] - temp["lat"].mean()) / temp["lat"].std()
lon = (temp["lon"] - temp["lon"].mean()) / temp["lon"].std()
lat = lat.broadcast_like(temp)
lon = lon.broadcast_like(temp)

temp, temp_bottom, chl, mixed, depth, month_sin, month_cos, lat, lon, zo, so, month, year, ugo, vgo, pp, cdm, spm, zsd = xr.align(
    temp, temp_bottom, chl, mixed, depth, month_sin, month_cos, lat, lon, zo, so, month, year, vgo, ugo, pp, cdm, spm, zsd, join="inner")

mask, fishing, fighins_cpue, maskB = xr.align(mask, fishing, fishing_cpue, maskB, join="inner")



croppedT = lambda da: da.sel(
    lon=slice(min_lon, max_lon),
    lat=slice(min_lat, max_lat),
    time=slice(min_time, max_time)
)

cropped = lambda da: da.sel(
    lon=slice(min_lon-1, max_lon+2),  ##modify to obtain diferent results
    lat=slice(min_lat-3, max_lat),
    time=slice(min_time, max_time)
)

fishing_target = croppedT(fishing)
fishing_cpue_target = croppedT(fishing_cpue)    
mask_target = croppedT(mask)
maskB_target = croppedT(maskB)

temp = cropped(temp)
temp_bottom = cropped(temp_bottom)
chl = cropped(chl)
mixed = cropped(mixed)
depth = cropped(depth)
zo = cropped(zo)
so = cropped(so)
month = cropped(month)
month_sin = cropped(month_sin)
month_cos = cropped(month_cos)
lat = cropped(lat)
lon = cropped(lon)
year = cropped(year)
ugo = cropped(ugo)
vgo = cropped(vgo)
pp = cropped(pp)
cdm = cropped(cdm)
spm = cropped(spm)
zsd = cropped(zsd)



In [12]:
y = fishing_cpue_target  # (time, lat, lon)
y = y.transpose("time", "lat", "lon")

m = mask_target
m = m.transpose("time", "lat", "lon")


vars_ = [temp, so, 
        ugo, vgo, chl, cdm,
        pp, temp_bottom, mixed, zo, 
        # month_sin, month_cos,
        # lat, lon,
        # depth,
        ]
vars_names = [v.name for v in vars_]
in_channels = len(vars_)
X = xr.concat(vars_, dim="channel")
X = X.transpose("time", "channel", "lat", "lon")


split_year = 2020

train_X = X.sel(time=slice(None, f"{split_year-1}-12-31"))
test_X  = X.sel(time=slice(f"{split_year}-01-01", None))

train_y = y.sel(time=slice(None, f"{split_year-1}-12-31"))
test_y  = y.sel(time=slice(f"{split_year}-01-01", None))

train_m = m.sel(time=slice(None, f"{split_year-1}-12-31"))
test_m  = m.sel(time=slice(f"{split_year}-01-01", None))


# -------- Check inflation factor --------
X_np = train_X.values  # (time, channel, lat, lon)
t, c, h, w = X_np.shape
X_flat = X_np.reshape(t, c, -1)       # (time, channel, pixels)
X_flat = X_flat.transpose(0, 2, 1)    # (time, pixels, channel)
X_flat = X_flat.reshape(-1, c)        # (samples, channel)
df = pd.DataFrame(X_flat, columns=vars_names)
vif_data = pd.DataFrame()
vif_data["feature"] = df.columns
vif_data["VIF"] = [variance_inflation_factor(df.values, i)for i in range(df.shape[1])]
print(vif_data.sort_values("VIF", ascending=False))

def create_windows(X, y, m, window=10):
    X_data = X.values   # (time, channels, H, W)
    y_data = y.values   # (time, H, W)
    m_data = m.values   # (time, H, W)  #this won't be used

    X_seq, y_seq, m_seq= [], [], []

    for i in range(len(X_data) - window):
        X_seq.append(X_data[i:i+window])
        y_seq.append(y_data[i+window])
        m_seq.append(m_data[i+window])
        

    return (
        torch.tensor(np.stack(X_seq), dtype=torch.float32),
        torch.tensor(np.stack(y_seq), dtype=torch.float32),
        torch.tensor(np.stack(m_seq), dtype=torch.float32),
    )

window = 12

X_train, y_train, m_train = create_windows(train_X, train_y, train_m, window)
X_test, y_test, m_test= create_windows(test_X, test_y, test_m, window)

print(X_train.shape)  # (N, T, C, H, W)
print(y_train.shape)  # (N, H, W)
print(m_train.shape)  # (N, H, W)
###### Data Loaders ######
batch_size = 24

train_loader = DataLoader(
    TensorDataset(X_train, y_train, m_train),
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    TensorDataset(X_test, y_test, m_test),
    batch_size=batch_size,
    shuffle=False
)


  feature        VIF
9      zo  11.531013
7    to_b  10.132864
4     CHL   7.647407
6      PP   7.428247
1      so   7.047682
3     ugo   3.197752
2     vgo   2.930630
5     CDM   2.322628
8  mlotst   2.242045
0      to   2.236450
torch.Size([72, 12, 10, 46, 32])
torch.Size([72, 27, 10])
torch.Size([72, 27, 10])


In [13]:
# -------- UNet 3D model--------

# -------- Basic Conv Block --------
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            # nn.LeakyReLU(0.1, inplace=True),
            nn.GELU(),
            nn.Dropout3d(0.2),
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            # nn.LeakyReLU(0.1, inplace=True),
            nn.GELU()
            )

    def forward(self, x):
        return self.block(x)


# -------- Decoder Block --------
class DecoderBlock3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_channels, out_channels, (1, 2, 2), stride=(1, 2, 2))
        self.conv_block = ConvBlock(out_channels + skip_channels, out_channels)
        
    def forward(self, x, skip):
        x = self.up(x)

        # Handle size mismatches
        x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
        x = torch.cat([x, skip], dim=1)

        x = self.conv_block(x)
        return x

# -------- Temporal Attention Block --------
class TemporalAttentionBlock3D(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.temporal_attn = nn.Sequential(
            nn.Conv3d(in_channels, in_channels, kernel_size=(3, 1, 1), padding=(1, 0, 0), groups=in_channels),
            nn.Sigmoid()
        )
        self.temporal_score = nn.Conv3d(in_channels, 1, kernel_size=1)
    
    def forward(self, x):
        # x shape: (B, C, T, H, W)
        attn = self.temporal_attn(x)
        feat = x * attn
        score = self.temporal_score(feat)
        weights = torch.softmax(score, dim=2) 
        feat = (feat * weights).sum(dim=2)    
        return feat

class GlobalAttentionBranch(nn.Module):
    def __init__(self, channels):
        super().__init__()
        reduction = max(channels // 4, 8)
        self.branch = nn.Sequential(
            nn.AdaptiveAvgPool3d((None, 1, 1)),
            nn.Conv3d(channels, reduction, 1),
            nn.ReLU(inplace=True),
            nn.Conv3d(reduction, channels, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.branch(x)

# -------- U-Net 3D --------
class UNet3D(nn.Module):
    def __init__(self, in_channels, out_hw, out_channels=1, base_ch=10):
        super().__init__()

        self.out_hw = out_hw  # (H, W)

        # -------- Encoder --------
        self.enc1 = ConvBlock(in_channels, base_ch)
        self.pool1 = nn.MaxPool3d((1,2,2))

        self.enc2 = ConvBlock(base_ch, base_ch*2)
        self.pool2 = nn.MaxPool3d((1,2,2)) #(1, 2, 2) for no temporal downsampling (2, 2, 2) for temporal downsampling


        # -------- Bottleneck --------
        self.bottleneck = ConvBlock(base_ch*2, base_ch*4)
        self.temp_mix = nn.Sequential(nn.Conv3d(base_ch*4, base_ch*4, kernel_size=(3,1,1), padding=(1,0,0)))
        self.global_att_branch = GlobalAttentionBranch(base_ch * 4)
        # -------- Temporal Attention --------
        self.temporal_attention = TemporalAttentionBlock3D(base_ch*4)

        # -------- Decoder --------
        self.decoder2 = DecoderBlock3D(base_ch*4, base_ch*2, base_ch*2)
        self.decoder1 = DecoderBlock3D(base_ch*2, base_ch, base_ch)
    

        # -------- Regression Head --------
        self.regressor = nn.Sequential(
            nn.Conv2d(base_ch, base_ch, 3, padding=1),
            nn.BatchNorm2d(base_ch),
            nn.GELU(),
            nn.Conv2d(base_ch, out_channels, 1)
        )
        

    def forward(self, x):
        # INPUT COMES AS: (B, T, C, H, W) Convert to:(B, C, T, H, W)
        
        x = x.permute(0, 2, 1, 3, 4)
        # -------- Encoder --------
        s1 = self.enc1(x)
        p1 = self.pool1(s1)

        s2 = self.enc2(p1)
        p2 = self.pool2(s2)

        # -------- Bottleneck --------
        b = self.bottleneck(p2)
        b = b + self.temp_mix(b)
        att = self.global_att_branch(b)
        b = b * att

        # temporal aggregation BEFORE decoder
        b = self.temporal_attention(b)  # (B,C,H,W)

        # fake temporal dim for decoder compatibility
        b = b.unsqueeze(2)
        s2 = s2.mean(dim=2, keepdim=True)
        s1 = s1.mean(dim=2, keepdim=True)

        # -------- Decoder --------
        d2 = self.decoder2(b, s2)
        d1 = self.decoder1(d2, s1)

        # final projection to output
        final = self.regressor(d1.squeeze(2))
        
        # interpolate to output shape      
        out = F.interpolate(final, size=self.out_hw, mode="bilinear", align_corners=False)

        return out
    

In [14]:
# ---------------CNN-Attention Regression Model------------------------------------------

# ---------------------------------------------------------
# Basic Conv Block
# ---------------------------------------------------------
class ConvBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.1),
            nn.Conv3d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x):
        return self.block(x)

# -------- Temporal Attention Block --------
class TemporalAttentionBlock3D(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.temporal_attn = nn.Sequential(
            nn.Conv3d(in_channels, in_channels, kernel_size=(3, 1, 1), padding=(1, 0, 0), groups=in_channels),
            nn.Sigmoid())
        self.temporal_score = nn.Conv3d(in_channels, 1, kernel_size=1)
    
    def forward(self, x):
        # x shape: (B, C, T, H, W)
        attn = self.temporal_attn(x)
        feat = x * attn
        score = self.temporal_score(feat)
        weights = torch.softmax(score, dim=2)  # Softmax over temporal dimension
        feat = (feat * weights).sum(dim=2)     # Aggregate over time
        return feat


# ---------------------------------------------------------
# CNN-Attention Regression Network
# ---------------------------------------------------------
class CNNAttention(nn.Module):

    def __init__(
        self,
        in_channels,
        out_hw,
        out_channels=1,
        base_ch=10
    ):
        super().__init__()

        self.out_hw = out_hw

        # =====================================================
        # Initial feature extraction
        # Equivalent to Conv_1 in the paper
        # =====================================================
        self.initial_conv = nn.Sequential(
            nn.Conv3d(in_channels, base_ch, kernel_size=3, padding=1),
            nn.BatchNorm3d(base_ch),
            nn.LeakyReLU(0.2, inplace=True)
        )

        # =====================================================
        # ATTENTION BRANCH
        # (Equivalent to GAP -> Fc_s -> Fc_e)
        # =====================================================
        self.gap = nn.AdaptiveAvgPool3d((1, 1, 1))

        reduction = max(base_ch // 2, 4)

        self.fc_s = nn.Sequential(
            nn.Conv3d(base_ch, reduction, kernel_size=1),
            nn.ReLU(inplace=True)
        )

        self.fc_e = nn.Sequential(
            nn.Conv3d(reduction, base_ch * 4, kernel_size=1),
            nn.Sigmoid()
        )

        # =====================================================
        # FEATURE EXTRACTION BRANCH
        # (Equivalent to Conv_2 and Conv_3)
        # =====================================================
        self.conv2 = nn.Sequential(
            nn.Conv3d(base_ch, base_ch * 2, kernel_size=3, padding=1),
            nn.BatchNorm3d(base_ch * 2),
            nn.ReLU(inplace=True)
        )

        self.conv3 = nn.Sequential(
            nn.Conv3d(base_ch * 2, base_ch * 4, kernel_size=3, padding=1),
            nn.BatchNorm3d(base_ch * 4),
            nn.ReLU(inplace=True)
        )

        # =====================================================
        # Fusion feature extraction
        # (Equivalent to Conv_4)
        # =====================================================
        self.conv4 = nn.Sequential(
            nn.Conv3d(base_ch * 4, base_ch * 8, kernel_size=3, padding=1),
            nn.BatchNorm3d(base_ch * 8),
            nn.ReLU(inplace=True)
        )

        # =====================================================
        # Temporal attention
        # Same style as your UNet model
        # =====================================================
        self.temporal_attn = TemporalAttentionBlock3D(base_ch * 8)

        # =====================================================
        # Regression Head
        # Replaces classifier head from original paper
        # =====================================================
        self.regressor = nn.Sequential(
            nn.Conv2d(base_ch * 8, base_ch * 4, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_ch * 4),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(base_ch * 4, out_channels, kernel_size=1)
        )

    # =========================================================
    # FORWARD
    # =========================================================
    def forward(self, x):

        # Input:
        # (B, T, C, H, W)
        #
        # Convert to:
        # (B, C, T, H, W)

        x = x.permute(0, 2, 1, 3, 4)

        # =====================================================
        # Initial features
        # =====================================================
        x0 = self.initial_conv(x)

        # =====================================================
        # Attention branch
        # =====================================================
        attn = self.gap(x0)

        attn = self.fc_s(attn)
        attn = self.fc_e(attn)

        # =====================================================
        # Feature branch
        # =====================================================
        feat = self.conv2(x0)
        feat = self.conv3(feat)

        # =====================================================
        # Attention fusion
        # Element-wise multiplication
        # =====================================================
        fused = feat * attn

        # =====================================================
        # Further feature extraction
        # =====================================================
        fused = self.conv4(fused)

        # =====================================================
        # Temporal attention
        # =====================================================

        feat = self.temporal_attn(fused)

        # =====================================================
        # Regression output
        # =====================================================
        out = self.regressor(feat)

        # Resize to desired output shape
        out = F.interpolate(out, size=self.out_hw,mode="bilinear",align_corners=False)

        return out

In [15]:
# -------- ConvGRU model --------
class ConvGRUCell(nn.Module):
    def __init__(self, in_ch, hidden_ch, kernel_size=3):
        super().__init__()
        padding = kernel_size // 2

        self.conv_zr = nn.Conv2d(in_ch + hidden_ch, 2 * hidden_ch, kernel_size, padding=padding)
        self.conv_h = nn.Conv2d(in_ch + hidden_ch, hidden_ch, kernel_size, padding=padding)

    def forward(self, x, h):
        combined = torch.cat([x, h], dim=1)

        zr = torch.sigmoid(self.conv_zr(combined))
        z, r = torch.chunk(zr, 2, dim=1)

        combined_reset = torch.cat([x, r * h], dim=1)
        h_tilde = torch.tanh(self.conv_h(combined_reset))

        h = (1 - z) * h + z * h_tilde
        return h
    


class ConvGRU(nn.Module):
    def __init__(self, in_channels, hidden=32, out_hw=(64,64)):
        super().__init__()

        self.out_hw = out_hw

        # spatial encoder (VERY simple)
        self.enc = nn.Sequential(
            nn.Conv2d(in_channels, hidden, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden, hidden, 3, padding=1),
            nn.ReLU()
        )

        self.gru = ConvGRUCell(hidden, hidden)

        self.dec = nn.Sequential(
            nn.Conv2d(hidden, hidden, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden, 1, 1)
        )

    def forward(self, x):
        # x: (B, T, C, H, W)
        B, T, C, H, W = x.shape

        h = torch.zeros(B, 32, H, W, device=x.device)

        for t in range(T):
            xt = self.enc(x[:, t])
            h = self.gru(xt, h)

        out = self.dec(h)
        out = F.interpolate(out, size=self.out_hw, mode="bilinear", align_corners=False)
        return out



In [16]:
# -------- LagStackCNN model--------

class LagStackCNN(nn.Module):
    def __init__(self, in_channels, out_hw, hidden=64):
        super().__init__()

        self.out_hw = out_hw
        self.in_channels = in_channels
        T = 12  ## harcoded window

        self.net = nn.Sequential(
            nn.Conv2d(T * in_channels, hidden, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden, hidden, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden, 1, 1)
        )

    def forward(self, x):
        # x: (B, T, C, H, W)
        B, T, C, H, W = x.shape

        x = x.reshape(B, T * C, H, W)

        out = self.net(x)
        out = F.interpolate(out, size=self.out_hw, mode="bilinear", align_corners=False)
        return out

In [ ]:
def masked_mse_loss(pred, target, mask, weight_factor=1):
    """
    pred:   (B, 1, H, W)
    target: (B, H, W)
    mask:   (B, H, W)
    """

    target = target.unsqueeze(1)   # -> (B,1,H,W)
    mask = mask.unsqueeze(1).float()
    weights = 1.0 + weight_factor * mask
    loss = (pred - target) ** 2
    loss = loss * weights
    return loss.mean()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
out_hw = y_train.shape[1:]
model = UNet3D(in_channels=in_channels, out_hw=out_hw).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)

best_test_r2 = -float("inf")
best_train_r2 = -float("inf")

patience = 50
epochs = 500
counter = 0

train_r2_history = []
val_r2_history = []

for epoch in range(epochs):

    # -------- Training --------
    model.train()

    total_loss = 0
    train_r2s = []
    train_maes = []
    train_biases = []
    train_corrs = []

    for x, y, m in train_loader:

        x = x.to(device)          # (B,T,C,H,W)
        y = y.to(device)          # (B,H,W)
        m = m.to(device)          # (B,H,W)

        optimizer.zero_grad()

        pred = model(x)           # (B,1,H,W)

        loss = masked_mse_loss(pred, y, m)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        pred_np = pred.detach().cpu().squeeze(1)   # (B,H,W)
        y_np = y.detach().cpu()                    # (B,H,W)
        mask_np = m.detach().cpu().bool()      # (B,H,W)

        y_true = y_np[mask_np].numpy()
        y_pred = pred_np[mask_np].numpy()

        if len(y_true) > 1:
            train_r2s.append(r2_score(y_true, y_pred))
            train_maes.append(mean_absolute_error(y_true, y_pred))
            train_biases.append(np.mean(y_pred - y_true))
            train_corrs.append(pearsonr(y_true, y_pred)[0])

    train_r2 = np.mean(train_r2s) if len(train_r2s) > 0 else float("nan")
    train_r2 = np.mean(train_r2s)
    train_mae = np.mean(train_maes)
    train_bias = np.mean(train_biases)
    train_corr = np.mean(train_corrs)


    # -------- Validation --------
    model.eval()

    test_r2s = []
    test_maes = []
    test_biases = []
    test_corrs = []

    with torch.no_grad():

        for x, y, m in test_loader:

            x = x.to(device)
            y = y.to(device)
            m = m.to(device)

            pred = model(x)

            pred_np = pred.detach().cpu().squeeze(1)   # (B,H,W)
            y_np = y.detach().cpu()                    # (B,H,W)
            mask_np = m.detach().cpu().bool()      # (B,H,W)

            y_true = y_np[mask_np].numpy()
            y_pred = pred_np[mask_np].numpy()

            if len(y_true) > 1:
                test_r2s.append(r2_score(y_true, y_pred))
                test_maes.append(mean_absolute_error(y_true, y_pred))
                test_biases.append(np.mean(y_pred - y_true))
                test_corrs.append(pearsonr(y_true, y_pred)[0])

    test_r2 = np.mean(test_r2s) if len(test_r2s) > 0 else float("nan")
    test_mae = np.mean(test_maes)
    test_bias = np.mean(test_biases)
    test_corr = np.mean(test_corrs)

    # -------- Early stopping --------
    improved = False

    if test_r2 > best_test_r2:
        best_test_r2 = test_r2
        improved = True

    if improved:
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print("Early stopping")
        break

    print(
    f"Epoch {epoch+1}/{epochs} | "
    f"Loss: {total_loss/len(train_loader):.4f} | "
    f"Train R2: {train_r2:.3f} | "
    f"Test R2: {test_r2:.3f} | Best Test R2: {best_test_r2:.3f} | MAE: {test_mae:.4f} | Bias: {test_bias:.4f} | Corr: {test_corr:.4f} | "
  
)

    train_r2_history.append(train_r2)
    val_r2_history.append(test_r2)

# -------- Plot --------
plt.figure(figsize=(8,5))

plt.plot(train_r2_history, label="Train R2")
plt.plot(val_r2_history, label="Val R2")

plt.xlabel("Epoch")
plt.ylabel("R2")
plt.title("Train vs Validation R2")
plt.annotate(f"Best Test R2: {best_test_r2:.4f}", xy=(0, best_test_r2), xytext=(-10, best_test_r2), textcoords="offset points")

plt.legend()
plt.grid(True)

plt.show()



Epoch 1/500 | Loss: 9.6790 | Train R2: -0.342 | Test R2: -0.466 | Best Test R2: -0.466 | MAE: 1.5613 | Bias: -1.5602 | Corr: 0.0382 | 
Epoch 2/500 | Loss: 9.1860 | Train R2: -0.273 | Test R2: -0.459 | Best Test R2: -0.459 | MAE: 1.5488 | Bias: -1.5477 | Corr: 0.3231 | 
Epoch 3/500 | Loss: 8.8524 | Train R2: -0.226 | Test R2: -0.451 | Best Test R2: -0.451 | MAE: 1.5360 | Bias: -1.5348 | Corr: 0.3897 | 
Epoch 4/500 | Loss: 8.4155 | Train R2: -0.165 | Test R2: -0.444 | Best Test R2: -0.444 | MAE: 1.5259 | Bias: -1.5248 | Corr: 0.4374 | 
Epoch 5/500 | Loss: 8.0746 | Train R2: -0.117 | Test R2: -0.439 | Best Test R2: -0.439 | MAE: 1.5214 | Bias: -1.5202 | Corr: 0.4748 | 
Epoch 6/500 | Loss: 7.5457 | Train R2: -0.041 | Test R2: -0.436 | Best Test R2: -0.436 | MAE: 1.5237 | Bias: -1.5225 | Corr: 0.4909 | 
Epoch 7/500 | Loss: 7.0277 | Train R2: 0.032 | Test R2: -0.436 | Best Test R2: -0.436 | MAE: 1.5356 | Bias: -1.5344 | Corr: 0.4971 | 
Epoch 8/500 | Loss: 6.7004 | Train R2: 0.079 | Test R2: 

In [2]:
fishing_cpue_target

NameError: name 'fishing_cpue_target' is not defined